In [ ]:
import torch
import occupycy

# -- Choose your device --
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
# device = torch.device("cpu")
print(f"Using device: {device}")

seed = 1
torch.manual_seed(seed);

In [ ]:
# 1. Simulate data (arbitrary non-linear functions)

true_params_psi = [-0.5, -0.5]
true_params_d = [1, -1, 0.02]

def true_f_psi(X_psi, params=true_params_psi): # here X_psi of shape (n_sites, n_cov_psi=2)
    return torch.sigmoid(
        params[0] * torch.cos(X_psi[:,0]) + params[1] * (X_psi[:,1] ** 2)
    ).unsqueeze(1)
def true_f_d(X_d, params=true_params_d): # here X_d of shape (n_sites, n_cov_d=1)
    return torch.sigmoid(
        torch.tanh(params[0] * X_d[:,0]) + params[1] * torch.cos(X_d[:,0])
    ).unsqueeze(1) * params[2]

y, visit_site_ids, site_ids_in_X_psi, X_psi, X_d, true_z = \
    occupycy.data.simulate_non_linear(
        n_sites=5000,
        n_cov_psi=2,
        n_cov_d=1,
        f_effort = lambda x: torch.full_like(x, 200),
        f_psi=true_f_psi,
        f_d=true_f_d,
        seed=seed,
        plot_summary=False,
        verbose=True
    )

In [ ]:
# 2. Build OccuData
data = occupycy.data.OccuData(
    y=y, 
    visit_site_ids=visit_site_ids,
    X_psi=X_psi,
    X_d=X_d,
    site_ids_in_X_psi=site_ids_in_X_psi
)
data.to(device=device)

In [ ]:
# DEFINE YOUR MODEL CLASS

# We will fit the true class of function with a custom model

# First we define each block of each model. 
# It will be psi_model and d_model attributes in the OccuModel.
# Each model block should be a torch.nn.Module to compute the LOGITS of the probabilities.

class CustomPsiModel(torch.nn.Module):

    def __init__(self):
        super().__init__()
        # Define your model (any architecture you want)
        # Here we just use individual parameters but you could use more complex architectures
        self.params = torch.nn.Parameter(torch.zeros(2)) # init 2 parameters to 0
        # You can also add other attributes that could be useful for your model
        self.very_important_attribute = "I am a very important attribute that can be used in the forward method if needed"
    
    # IMPORTANT: the forward returns the LOGIT of the occupancy probability, not the occupancy probability itself.
    def forward(self, X_psi):
        """
        Model for the logit of the occupancy probability.

        Parameters
        ----------
        X_psi : torch.Tensor
            Tensor of shape (n_sites, n_cov_psi) with the covariates for the occupancy model. Here n_cov_psi=2.
        
        Returns
        -------
        torch.Tensor
            Tensor of shape (n_sites, n_species) with the LOGIT of the occupancy probability for each site. Here n_species=1.
        """
        cos_term = self.params[0] * torch.cos(X_psi[:,0]) # here we use the first parameter for the cosine term
        quad_term = self.params[1] * (X_psi[:,1] ** 2) # here we use the second parameter for the quadratic term
        out = cos_term + quad_term # logit of the occupancy probability, shape (n_sites,)
        out = out.unsqueeze(1) # shape (n_sites, 1) to match the expected output shape (n_sites, n_species)
        return out
    
class CustomDModel(torch.nn.Module):

    def __init__(self, n_params):
        super().__init__()
        # Same for the detection model, define your architecture and parameters
        self.params = torch.nn.Parameter(torch.ones(n_params)) # init to 1

    # IMPORTANT: the forward returns the LOGIT of the detection probability, not the detection probability itself.
    # We make it shorter here by directly using the true function, but you can of course use a different architecture and model.
    def forward(self, X_d):
        return torch.logit(true_f_d(X_d, params=self.params))
    
# Now we can define the custom OccuModel
class CustomModel(occupycy.models.OccuModel):
    
    def __init__(self, n_params_psi, n_params_d):
        # Define the number of species in the super init, here 1
        super().__init__(n_species=1)
        # MANDATORY: define the psi_model and d_model attributes, which should be torch.nn.Module with a forward method
        self.psi_model = CustomPsiModel() # This one does not take any argument
        self.d_model = CustomDModel(n_params=n_params_d) # This one takes the number of parameters as argument

In [ ]:
# 3. Define a model (linear)
model = CustomModel(n_params_psi=len(true_params_psi), n_params_d=len(true_params_d))
model.to(device=device)

# 4. Fit the model

# # > Adam
# res = occupycy.fit.fit_sgd_like(
#     data,
#     model,
#     n_epochs=1000,
#     val_fraction=0.15,
#     batch_size=512,
#     optimizer_psi_cls=torch.optim.Adam,
#     optimizer_d_cls=torch.optim.Adam,
#     lr_psi=1e-2,
#     lr_d=1e-2,
#     patience_early_stopping=100,
#     patience_reduceLROnPlateau=20,
#     device=device
# )

# > L-BFGS full batch
res = occupycy.fit.fit_lbfgs_full_batch(
    data,
    model,
    max_iter=1000,
    lr=0.05,
    device=device
)

In [ ]:
# Explore the parameters
for i in range(len(true_params_psi)):
    print(f"param_psi{i}: {model.psi_model.params[i].item():.3f} (true value: {true_params_psi[i]})")
for i in range(len(true_params_d)):
    print(f"param_d{i}: {model.d_model.params[i].item():.3f} (true value: {true_params_d[i]})")

In [ ]:
# Plot the losses
import matplotlib.pyplot as plt

loss_train = res.history["train_loss"]
loss_val = res.history["val_loss"]
epochs = range(1, len(loss_train) + 1)
skip_initial_epochs = min(0, len(epochs) // 2)

fig, ax1 = plt.subplots(figsize=(6, 3))
ax2 = ax1.twinx()
l1, = ax1.plot(epochs[skip_initial_epochs:], loss_train[skip_initial_epochs:], label="train", color="blue")
lines = [l1]
if len(loss_val) > 0:
    l2, = ax2.plot(epochs[skip_initial_epochs:], loss_val[skip_initial_epochs:], label="val loss", color="orange")
    lines.append(l2)
ax1.set_xlabel("Epoch")
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper right')

ax1.set_xscale("log")
ax2.set_xscale("log")
ax1.set_yscale("log")
ax2.set_yscale("log")

plt.title("Losses")
plt.show()